# Controller Performance Metrics – Full Reference

This document describes every performance index computed by `ControllerPerformance`
(module `metrics.py`).  
The data come from a simulation history CSV (produced by `robot_demo.py`).  
All tunable thresholds are exposed as class constants; the default values are given.

---

## 1. Navigation Efficiency

These indices measure how directly and quickly the robot reaches its moving goal.

### 1.1 Path Length (m)

$$
PL = \sum_{i=1}^{N-1} \| \mathbf{p}_{i+1} - \mathbf{p}_i \|
$$

**Data source** – `self.robot_pos` (robot $(x,y)$ at each simulation step).  
**Utility** – A short path indicates efficient navigation; a long path suggests detours or wandering.

### 1.2 Path Efficiency

$$
PE = \frac{ \| \mathbf{p}_{\text{goal}} - \mathbf{p}_{\text{start}} \| }{ PL }
$$

**Utility** – Values near 1.0 mean the robot moved almost straight to its final position; lower values indicate meandering.  
*(The inverse of tortuosity.)*

### 1.3 Time to Goal (s)

$$
TTG = \begin{cases}
t_{\text{first reach}} & \text{if } d(t) < \text{goal\_tolerance} \\
t_{\text{end}} & \text{otherwise}
\end{cases}
$$

**Utility** – Measures how early the robot entered the goal region; if never reached, the total simulation time is returned.

### 1.4 Average Speed (m/s)

$$
\bar v = \frac{1}{N} \sum_{i=0}^{N-1} v_i
$$

**Data** – `self.robot_speed` (from `lin_speed` or $ \|\mathbf{v}\| $).  
**Utility** – Higher average speed can indicate more aggressive navigation; must be balanced with safety.

### 1.5 Max Speed (m/s)

$$
v_{\max} = \max_i v_i
$$

**Utility** – Peak speed; controllers that overshoot are penalised in other metrics.

### 1.6 Speed Variance

$$
\sigma_v^2 = \frac{1}{N}\sum_{i=0}^{N-1}(v_i - \bar v)^2
$$

**Utility** – Large variance means the robot alternates between fast and slow; a smoother ride has low variance.

### 1.7 Stop Ratio

$$
\text{stop\_ratio} = \frac{1}{N}\sum_{i=0}^{N-1} \mathbb{1}[v_i < v_{\text{thresh}}]
$$

($v_{\text{thresh}} = 0.05\ \text{m/s}$)  
**Utility** – High values indicate the robot often gets stuck; a good controller should rarely stop.

---

## 2. Safety & Collision

These indices evaluate whether the robot keeps a safe distance and avoids actual collisions.

### 2.1 Collision Events (distinct)

Events are counted using a cooldown (0.5 s) – if the robot repeatedly violates the safety margin within a short time, only the first violation is counted as one event.  
This matches the analysis printed at the end of `robot_demo.py`.

**Utility** – A lower number is better; zero means the robot never once entered the safety zone.

### 2.2 Collision Steps

The raw number of simulation steps where `in_collision` was `True`.  
**Utility** – Shows how long the robot remained inside the safety zone; high values indicate prolonged unsafe behavior.

### 2.3 Minimum Distance to Obstacles (m)

$$
d_{\min} = \min_{i} \min_{o \in \mathcal{O}_i} \bigl( \| \mathbf{p}_{r,i} - \mathbf{p}_{o,i} \| - r_o - r_{\text{robot}} \bigr)
$$

**Utility** – Negative values indicate an overlap; positive values show the safety margin. The higher the better.

### 2.4 Social Safety Index (SSI) – Personal Space Intrusions

$$
SSI = \frac{1}{N} \sum_{i=0}^{N-1} \Bigl[ \exists\ \text{pedestrian } p : \| \mathbf{p}_{r,i} - \mathbf{p}_{p,i} \| < d_{\text{personal}} \Bigr]
$$

**Utility** – Fraction of time the robot is closer than $d_{\text{personal}}$ (default 1.5 m) to any pedestrian.  
Low values are better; zero means the personal space was never violated.

### 2.5 Personal Space Intrusion Events

Count of distinct time intervals where any pedestrian enters $d_{\text{personal}}$, using the same cooldown logic as collision events.  
**Utility** – Indicates how often the robot entered the “social bubble” of a pedestrian.

### 2.6 Minimum Time to Collision (s)

$$
TTC = \min_{i} \min_{p} \frac{ \| \mathbf{p}_{r,i} - \mathbf{p}_{p,i} \| }{ \| \mathbf{v}_{r,i} - \mathbf{v}_{p,i} \| }
$$

(under constant‑velocity assumption; capped at `ttc_time_horizon` = 2.0 s).  
**Utility** – A higher value means the robot had more time to react; low values indicate dangerous situations.

---

## 3. Social Comfort

Measures how naturally and considerately the robot behaves around people.

### 3.1 Social Individual Index (SII) – Comfort Zone Intrusion

$$
SII = \frac{1}{N} \sum_{i=0}^{N-1} \sum_{p} \max\!\left(0,\; 1 - \frac{ \| \mathbf{p}_{r,i} - \mathbf{p}_{p,i} \| }{ d_{\text{comfort}} } \right)
$$

**Utility** – Accumulated intrusion into the comfort zone ($d_{\text{comfort}}=3.0$ m).  
A lower value means the robot stays well outside the comfort zone.

### 3.2 Relative Motion Index (RMI)

$$
RMI = \frac{1}{N} \sum_{i=1}^{N-1} \sum_{p} \frac{ \max\!\bigl(0,\; (\mathbf{v}_r - \mathbf{v}_p) \cdot \mathbf{n}_{rp} \bigr) }{ \| \mathbf{p}_r - \mathbf{p}_p \| }
$$

**Utility** – Penalises rapid approaches toward pedestrians.  
Lower is better; zero means the robot never moved toward a pedestrian.

### 3.3 Social Grace Index (SGI)

$$
SGI = \frac{1}{\text{\#scores}} \sum \cos(\theta_{\text{robot}} - \theta_{\text{to ped}})
$$

(only for pedestrians between 0.5 m and 3.0 m).  
**Utility** – Measures whether the robot’s orientation is socially polite: a value near +1 means the robot “faces” the pedestrian (like a person would), while −1 indicates turning away.  
It quantifies the *naturalness* of the approach.

---

## 4. Smoothness & Jerk

Physical comfort for the robot (and indirectly for observers) is captured by these indices.

### 4.1 Acceleration RMS (m/s²)

$$
a_{\text{rms}} = \sqrt{ \frac{1}{N} \sum a_{\text{lin}}^2 }
$$

**Utility** – High values mean jerky starts/stops; smooth controllers have low $a_{\text{rms}}$.

### 4.2 Max Acceleration (m/s²)

$$
a_{\max} = \max_i |a_{\text{lin},i}|
$$

**Utility** – The peak acceleration experienced by the robot.

### 4.3 Jerk RMS (m/s³)

$$
j_{\text{rms}} = \sqrt{ \frac{1}{N} \sum j_{\text{lin}}^2 }
$$

**Utility** – Rate of change of acceleration; smooth controllers have low jerk.

### 4.4 Jerk Mean (m/s³)

$$
j_{\text{mean}} = \frac{1}{N-1} \sum_{i=1}^{N-1} |j_i|
$$

**Utility** – Another measure of jerk; complements RMS.

### 4.5 Smoothness (rad)

$$
S = \sum_{i=1}^{N-2} | \Delta \theta_i |,\quad
\Delta \theta_i = \text{normalised heading difference}
$$

**Utility** – Sum of absolute heading changes; a smaller sum means a straighter path.

### 4.6 Sinuosity (rad)

Standard deviation of the step‑to‑step heading changes $|\Delta\theta_i|$.  
**Utility** – High variability indicates erratic turning.

---

## 5. Path Quality

Indices describing the geometric properties of the trajectory.

### 5.1 Legibility

$$
L = \frac{1}{N} \sum \cos( \theta_i - \theta_{\text{goal},i} )
$$

**Utility** – How well the robot points toward the goal; high values (>0.9) mean the intention is clear to an observer.

### 5.2 Direction Changes

Count of heading changes larger than $5^\circ$.  
**Utility** – Frequent small turns indicate indecisiveness; a good path has few direction changes.

### 5.3 Directness ( = Path Efficiency)

Already defined in §1.2.

### 5.4 Tortuosity

$$
\tau = \frac{PL}{\|\mathbf{p}_{\text{end}}-\mathbf{p}_{\text{start}}\|}
$$

**Utility** – $\tau=1$ for a straight line; higher values mean a winding path.

### 5.5 Mean Curvature (1/m)

$$
\bar\kappa = \frac{1}{M} \sum \kappa_i,\quad
\kappa_i = \frac{|x'_i y''_i - y'_i x''_i|}{(x_i'^2+y_i'^2)^{3/2}}
$$

(computed via central differences).  
**Utility** – Average sharpness of turns; lower is smoother.

### 5.6 Max Curvature (1/m)

$$
\kappa_{\max} = \max_i \kappa_i
$$

**Utility** – The sharpest turn the robot executed.

---

## 6. Computational Performance

Real‑time capability of the controller.

### 6.1 Average Computation Time (ms)

$$
\bar t_{\text{comp}} = \frac{1}{N_{\text{control}}} \sum t_{\text{comp},k}
$$

### 6.2 Real‑Time Factor

$$
RTF = \frac{\Delta t_{\text{sim}}}{\bar t_{\text{comp}}}
$$

($>1$ means faster than real time).

---

## Default Thresholds (class constants)

| Parameter | Value | Description |
|-----------|-------|-------------|
| `GOAL_TOLERANCE` | 0.25 m | Goal reached if distance below |
| `SAFETY_MARGIN` | 1.0 m | Clearance for collision detection |
| `PERSONAL_SPACE` | 1.5 m | Distance for SSI and intrusion events |
| `COMFORTABLE_DISTANCE` | 3.0 m | Comfort zone for SII |
| `CRITICAL_DISTANCE` | 0.3 m | Near‑miss threshold |
| `TTC_TIME_HORIZON` | 2.0 s | Upper bound for TTC |
| `STOP_SPEED_THRESH` | 0.05 m/s | Below this, robot is considered stopped |
| `DIRECTION_CHANGE_THRESH` | 5 deg | Minimum heading change to count as a direction change |

All thresholds can be overridden when creating a `ControllerPerformance` instance.

---

## Data Sources in the History CSV

- `agent_type` – `"robot"`, `"user"`, `"pedestrian"`, `"static_obstacle"`, `"dynamic_obstacle"`  
- `x`, `y`, `vx`, `vy`, `theta`, `radius` – pose, velocity, size  
- `lin_speed`, `ang_speed` – robot’s actual speed  
- `lin_accel`, `lin_jerk` – estimated linear acceleration and jerk (from the Kalman filter)  
- `goal_x`, `goal_y` – target goal position at each robot step  
- `in_collision` – boolean collision flag  
- `comp_time` – computation time for that control step  

The `ControllerPerformance` class pre‑processes these columns into NumPy arrays for efficient metric calculation.

# Controller Performance Rankings

**Weighting per category:**  
- **Computational:** ×3  
- **Smoothness & Jerk:** ×3  
- **Safety & Collision:** ×2  
- **Social Comfort:** ×2  
- **Path Quality:** ×1  
- **Navigation Efficiency:** ×1  

**Controllers evaluated:**  
- **MPC‑Based (7):** MPCCMPPI, DCBFMPCCMPPI, DCBFNMPC, RiskAwareMPPI, DCBFMPPI, MPPI, NMPC  
- **DWA Variants (5):** BasicDWA, DW4DO, DWA_VO, DWA_ORCA, DWA_RVO  
- **Overall (13):** All above + SFMController  

**Ranking method:**  
For each metric, controllers are sorted from best to worst. The best receives *N* points, the worst receives 1 (ties share the average of the ranks they span).  
Points are multiplied by the category weight, then summed across all 29 metrics to produce the final score.

---

## 1. MPC‑Based Controllers (7 controllers)

| Rank | Controller | Score |
|------|-----------|-------|
| 1 | MPCCMPPIController | **322.0** |
| 2 | RiskAwareMPPIController | **217.0** |
| 3 | MPPIController | **214.0** |
| 4 | DCBFMPPIController | **212.5** |
| 5 | DCBFMPCCMPPIController | **204.0** |
| 6 | DCBFNMPCController | **200.5** |
| 7 | NMPCController | **142.0** |

**Why:**  
- MPCC‑MPPI dominates because of perfect safety (zero collisions, zero intrusions) and strong smoothness / path quality.  
- RiskAwareMPPI benefits from excellent path efficiency and the best smoothness, offsetting moderate computation and safety scores.  
- MPPI and DCBFMPPI are very close; MPPI’s better time‑to‑goal and average speed give it a slight edge.  
- DCBFNMPC is dragged down by extremely high computation time and negative Social Grace Index, despite good path tracking.  
- NMPC (without DCBF) is last due to poor safety and worst computation time.

---

## 2. DWA Variants (5 controllers)

| Rank | Controller | Score |
|------|-----------|-------|
| 1 | BasicDWA | **207.0** |
| 2 | DWA_VO | **170.5** |
| 3 | DW4DO | **161.5** |
| 4 | DWA_ORCA | **142.0** |
| 5 | DWA_RVO | **129.0** |

**Why:**  
- BasicDWA leads because of shorter path length, better path efficiency, and moderate collisions.  
- DWA_VO and DW4DO are similar, but DWA_VO pulls ahead on computation time.  
- DWA_ORCA and DWA_RVO have zero collisions but extremely poor path efficiency, very high direction changes, and poor social comfort (low SGI), which outweigh their safety advantage.

---

## 3. Overall Ranking (all 13 controllers)

| Rank | Controller | Score |
|------|-----------|-------|
| **1** | MPCCMPPIController | **565.0** |
| **2** | SFMController | **467.0** |
| **3** | BasicDWA | **443.0** |
| **4** | RiskAwareMPPIController | **401.5** |
| **5** | DCBFMPPIController | **384.0** |
| **6** | MPPIController | **366.5** |
| **7** | DCBFMPCCMPPIController | **365.0** |
| **8** | DCBFNMPCController | **362.5** |
| **9** | DWA_VO | **354.0** |
| **10** | DW4DO | **345.0** |
| **11** | DWA_ORCA | **318.5** |
| **12** | DWA_RVO | **302.5** |
| **13** | NMPCController | **282.0** |

**Key takeaways:**  
- **MPCC‑MPPI** is the clear winner – perfect safety, excellent path quality, strong smoothness, and good computational performance.  
- **SFM** surprises at 2nd place, driven by unbeatable computation time and outstanding smoothness/jerk values, coupled with decent navigation and path quality.  
- **BasicDWA** is the best DWA variant, ranking 3rd overall thanks to good safety and very fast computation.  
- **RiskAwareMPPI** performs consistently well across all categories, securing 4th.  
- **DCBFMPPI**, **MPPI**, **DCBFMPCCMPPI**, and **DCBFNMPC** are tightly clustered – their scores differ by less than 22 points, indicating comparable overall performance.  
- **NMPC** (no CBF) ranks last due to its extreme computation time (311 ms) and worst‑in‑class collision record.  
- **DWA_ORCA and DWA_RVO** trail because their excellent safety cannot compensate for very poor path efficiency and social behaviour.